In [3]:
import pandas as pd
import re
import pickle
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

df = pd.read_csv(r'C:\Users\ans\Desktop\CS-604 FYP\FYP-Project\train_data.csv')
df = df[['reviews.text', 'sentiment']].dropna()
df = df.rename(columns={'reviews.text': 'review', 'sentiment': 'label'})

label_mapping = {'Positive': 1, 'Negative': -1, 'Neutral': 0}
df['label'] = df['label'].map(label_mapping)

df.head()

,review,label
0,Purchased on Black FridayPros - Great Price (e...,1
1,I purchased two Amazon in Echo Plus and two do...,1
2,Just an average Alexa option. Does show a few ...,0
3,"very good product. Exactly what I wanted, and ...",1
4,This is the 3rd one I've purchased. I've bough...,1


In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text)).lower()
    words = word_tokenize(text)
    cleaned = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(cleaned)

df = df.sample(n=10000, random_state=42) if len(df) > 10000 else df
df['cleaned_review'] = df['review'].apply(clean_text)

df[['review', 'cleaned_review']].head()

,review,cleaned_review
0,Purchased on Black FridayPros - Great Price (e...,purchased black fridaypros great price even sa...
1,I purchased two Amazon in Echo Plus and two do...,purchased two amazon echo plus two dot plus fo...
2,Just an average Alexa option. Does show a few ...,average alexa option show thing screen still l...
3,"very good product. Exactly what I wanted, and ...",good product exactly wanted good price
4,This is the 3rd one I've purchased. I've bough...,rd one ive purchased ive bought one niece case...


In [5]:
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_review'], df['label'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_acc = accuracy_score(y_test, lr_model.predict(X_test_tfidf))
print(f"Logistic Regression Accuracy: {lr_acc * 100:.2f}%")

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)
svm_acc = accuracy_score(y_test, svm_model.predict(X_test_tfidf))
print(f"SVM Accuracy: {svm_acc * 100:.2f}%")

Logistic Regression Accuracy: 94.12%
SVM Accuracy: 95.62%


In [6]:
import warnings
import pickle
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

dl_model = MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42)
dl_model.fit(X_train_tfidf, y_train)
dl_acc = accuracy_score(y_test, dl_model.predict(X_test_tfidf))
print(f"Deep Learning Accuracy: {dl_acc * 100:.2f}%")

with open('svm_model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)
    
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

Deep Learning Accuracy: 95.25%
